# Agno

[Agno](agno.com) es un framework que pretende aligerar la forma en la que otros frameworks nos obligan a trabajar para crear nuestros agentes. Simplifica gran parte del código a generar centrándose en los aspectos clave. Ofrece una plataforma donde a futuro será posible gestionar nuestros agentes y flujos de trabajo (https://app.agno.com/) aunque de momento lo podemos emplear como framework local. Dispone de multitud de ejemplos en la [documentación](https://docs.agno.com/).

![](https://mintcdn.com/agno/QZOB15dhrj4yAmBd/images/workspace.png?w=840&maxW=3034&auto=format&n=QZOB15dhrj4yAmBd&q=85&s=192feab94035c340f257f7b7f228cd19)

Podemos levantar todo un workspace local con ejemplos siguiendo los pasos en: https://docs.agno.com/workspaces/introduction

Aunque para estos ejemplos iremos paso a paso, empezando con la importación de las claves de conexión y algunos parámetros básicos.

In [1]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [3]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat

# Asegúrate de que tu clave OPENAI_API_KEY esté configurada en el entorno

# Crear el asistente de matemáticas
agent = Agent(
    model=OpenAIChat(id="gpt-4o-mini", temperature=0),
    instructions="Eres un asistente de matemáticas.",
    markdown=True,  # Mantén markdown si tu versión lo soporta
)

# Ejemplo de uso
agent.print_response(
    "Hola, ¿puedes ayudarme con unos cálculos? ¿Cuánto es 1234 * 56?", stream=True
)


Output()

In [ ]:
agent.print_response(
    "¿Cuanto es 4 * 5?", stream=True
)

Al igual que LangChain podemos trazar la actividad pero veréis que en este caso es algo menos visual en [LangSmith via OpenTelemetry](https://docs.agno.com/examples/concepts/observability/langsmith-via-openinference)

In [ ]:
import os
from openinference.instrumentation.agno import AgnoInstrumentor
from opentelemetry import trace as trace_api
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor

# Set the endpoint and headers for LangSmith
endpoint = "https://api.smith.langchain.com/otel/v1/traces"
headers = {
    "x-api-key": os.getenv("LANGSMITH_API_KEY"),
    "Langsmith-Project": os.getenv("LANGSMITH_PROJECT"),
}

# Configure the tracer provider
tracer_provider = TracerProvider()
tracer_provider.add_span_processor(
    SimpleSpanProcessor(OTLPSpanExporter(endpoint=endpoint, headers=headers))
)
trace_api.set_tracer_provider(tracer_provider=tracer_provider)

# Start instrumenting agno
AgnoInstrumentor().instrument()

Podemos extender el uso de herramientas tal y como vimos con LangChain.

In [5]:
def multiply(a: int, b: int) -> int:
    """Multiply a and b.

    Args:
        a: first int
        b: second int
    """
    return a * b

In [ ]:
# Crear un agente con modelo OpenAI
agent = Agent(
    model=OpenAIChat(id="gpt-4o-mini", temperature=0), 
    instructions="Eres un asistente de matemáticas.",
    tools=[multiply],
    markdown=True,  # Mantén esto si quieres respuestas en formato Markdown
)

# Ejemplo de uso
agent.print_response(
    "¿Cuánto es 4 * 5?", stream=True
)

Output()

Podemos incluir herramientas más elaboradas como los buscadores web.

In [9]:
from agno.tools.tavily import TavilyTools

# Search engine
search = TavilyTools()

agent = Agent(
    model=OpenAIChat(id="gpt-4o-mini", temperature=0), 
    instructions="""
        Eres un especialista en viajes trabajando en una agencia. 
        Tu trabajo es proponer opciones de viajes a los clientes aunque tienes una predilección por Bilbao y siempre que puedes lo propones como destino.""",
    tools=[search],
    markdown=True,
)

agent.print_response(
    "¿Qué viajes hay a Madrid desde Santander?", stream=True
)

Output()

In [10]:
agent.print_response(
    "Está bien, busca entonces vuelos a ese destino", stream=True
)

Output()

Acordémonos de que los agentes no tienen memoria. Necesitamos indicarle cuantos mensajes debe recordar.

En muchos casos necesitaremos que nuestro interlocutor nos de el ok a la operación. Para eso, debemos instruir al agente de que la ejecución de la herramienta debe disponer de una aceptación.

In [13]:
import json
from typing import Any, Callable, Dict, Iterator

import httpx
from agno.exceptions import StopAgentRun
from agno.tools import tool
from rich.console import Console
from rich.prompt import Prompt

# Consola
console = Console()

# Confirmation hook
def confirmation_hook(
    function_name: str, function_call: Callable, arguments: Dict[str, Any]
):
    # Get the live display instance from the console
    live = console._live

    # Stop the live display temporarily so we can ask for user confirmation
    live.stop()  # type: ignore

    # Ask for confirmation
    console.print(f"\nVoy a ejecutar [bold blue]{function_name}[/]")
    message = (
        Prompt.ask("¿Quieres que proceda?", choices=["s", "n"], default="s")
        .strip()
        .lower()
    )

    # Restart the live display
    live.start()  # type: ignore

    # If the user does not want to continue, raise a StopExecution exception
    if message != "s":
        raise StopAgentRun(
            "Tool call cancelled by user",
            agent_message="Stopping execution as permission was not granted.",
        )
    
    # Call the function
    result = function_call(**arguments)

    # Optionally transform the result

    return result

# A tool that requests confirmation
@tool(tool_hooks=[confirmation_hook])
def get_top_hackernews_stories(num_stories: int) -> Iterator[str]:
    """Fetch top stories from Hacker News.

    Args:
        num_stories (int): Number of stories to retrieve

    Returns:
        str: JSON string containing story details
    """
    # Fetch top story IDs
    response = httpx.get("https://hacker-news.firebaseio.com/v0/topstories.json")
    story_ids = response.json()

    # Yield story details
    final_stories = []
    for story_id in story_ids[:num_stories]:
        story_response = httpx.get(
            f"https://hacker-news.firebaseio.com/v0/item/{story_id}.json"
        )
        story = story_response.json()
        if "text" in story:
            story.pop("text", None)
        final_stories.append(story)

    return json.dumps(final_stories)

In [15]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat 

# Create an Agent
agent = Agent(
    model=OpenAIChat(id="gpt-4o-mini", temperature=0), 
    instructions="Eres un especialista en periodismo tecnológico y puedes predecir tendencias de mercado basado en noticias de hackernews.",
    tools=[get_top_hackernews_stories],
    markdown=True,
)

agent.print_response(
    "¿Qué disrupciones prevés para este final de año?", stream=True, console=console
)

Output()

Voy a ejecutar get_top_hackernews_stories

¿Quieres que proceda? [s/n] (s):